# GraviLearn - Machine learning pipelines

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviLearn;
using Gravicode.Science.GraviLearn.Clustering;
using Gravicode.Science.GraviLearn.Decomposition;
using Gravicode.Science.GraviLearn.ModelSelection;
using Gravicode.Science.GraviLearn.Preprocessing;
using Gravicode.Science.GraviLearn.Trees;
using Gravicode.Science.GraviNum;

var iris = Datasets.LoadIris();
Console.WriteLine($"{iris.SampleCount} samples, {iris.FeatureCount} features, {iris.TargetNames.Count} classes");

## Train a random forest

In [ ]:
var split = Selection.Split(iris.Features, iris.Target, testSize: 0.3, seed: 42, stratify: true);

var forest = new RandomForestClassifier(nTrees: 200, seed: 42);
forest.Fit(split.TrainX, split.TrainY);

Console.WriteLine($"test accuracy       : {forest.Score(split.TestX, split.TestY):P2}");
Console.WriteLine($"out-of-bag accuracy : {forest.OutOfBagScore:P2}");

## Classification report

In [ ]:
var predictions = forest.Predict(split.TestX);
Console.WriteLine(Metrics.ClassificationReport(split.TestY, predictions, iris.LabelNames));

## Confusion matrix

In [ ]:
var digits = Datasets.LoadDigits();
var digitSplit = Selection.Split(digits.Features, digits.Target, testSize: 0.3, seed: 42, stratify: true);

var pipeline = new Pipeline()
    .Add(new StandardScaler())
    .Add(new PCA(components: 30))
    .Add(new Gravicode.Science.GraviLearn.Neighbors.KNearestNeighborsClassifier(k: 3));
pipeline.Fit(digitSplit.TrainX, digitSplit.TrainY);

var confusion = Metrics.ConfusionMatrix(digitSplit.TestY, pipeline.Predict(digitSplit.TestX));

var plot = new ScottPlot.Plot();
var heatmap = plot.Add.Heatmap(confusion.To2DArray());
heatmap.Colormap = new ScottPlot.Colormaps.Viridis();
plot.Add.ColorBar(heatmap);
plot.Title($"digits confusion matrix - {pipeline.Score(digitSplit.TestX, digitSplit.TestY):P1} accuracy");
plot.XLabel("predicted"); plot.YLabel("true");
plot.GetPngHtml(800, 700)

## Compare models with cross-validation

In [ ]:
var candidates = new (string, Func<IEstimator>)[]
{
    ("LogisticRegression", () => new Gravicode.Science.GraviLearn.Linear.LogisticRegression(0.3, 1500)),
    ("DecisionTree", () => new DecisionTree(maxDepth: 4)),
    ("RandomForest", () => new RandomForestClassifier(nTrees: 100, seed: 42)),
    ("kNN", () => new Gravicode.Science.GraviLearn.Neighbors.KNearestNeighborsClassifier(k: 5)),
};

foreach (var (name, factory) in candidates)
{
    var result = Selection.CrossValidate(factory, iris.Features, iris.Target, folds: 5, stratified: true, seed: 42);
    Console.WriteLine($"{name,-22}{result}");
}

## Principal components and clustering

In [ ]:
var scaled = new StandardScaler().FitTransform(iris.Features);
var pca = new PrincipalComponentAnalysis(2);
var projected = pca.FitTransform(iris.Features);

Console.WriteLine($"variance explained: PC1 {pca.ExplainedVarianceRatio.At(0):P2}, PC2 {pca.ExplainedVarianceRatio.At(1):P2}");

var plot2 = new ScottPlot.Plot();
foreach (var group in Enumerable.Range(0, iris.SampleCount).GroupBy(i => (int)iris.Target.At(i)))
{
    var xs = group.Select(i => projected[i, 0]).ToArray();
    var ys = group.Select(i => projected[i, 1]).ToArray();
    var scatter = plot2.Add.ScatterPoints(xs, ys);
    scatter.LegendText = iris.TargetNames[group.Key];
    scatter.MarkerSize = 8;
}
plot2.Title("Iris projected onto its first two principal components");
plot2.ShowLegend();
plot2.GetPngHtml(850, 600)

## Permutation importance

Shuffle a column and watch accuracy fall: the drop is what that column's relationship with the
target was worth. Run it on **held-out** data — on the training set it measures memorisation.

It also measures something different from a tree's built-in importances, which describe how the tree
was *built* and are known to favour high-cardinality features regardless of whether they predict.


In [ ]:
using Gravicode.Science.GraviLearn.Explain;
using Gravicode.Science.GraviNum;

var explainForest = new RandomForestClassifier(nTrees: 60, seed: 42);
explainForest.Fit(split.TrainX, split.TrainY);

foreach (var importance in PermutationImportance.Ranked(explainForest, split.TestX, split.TestY, repeats: 8))
    Console.WriteLine($"{iris.FeatureNames[importance.Feature],-16} {importance.Mean,7:F4} +/- {importance.StandardDeviation:F4}");

Console.WriteLine("\nPetal measurements dominate, which is the known structure of this dataset.");
Console.WriteLine("Correlated features share the blame and each looks less important than it is -");
Console.WriteLine("that is a real limitation of the method, and the spread is what warns you.");


## Shapley values

A different question: not "which features does this model rely on" but "why did it say *that*, for
*this* row". The two routinely disagree — a feature can be globally unimportant and decisive for one
case.

Explaining the **probability** rather than the hard label: a class index is a step function, and
attributing changes in a step tells you far less than attributing the confidence behind it.


In [ ]:
NdArray AsRow(NdArray vector)
{
    var matrix = NdArray.Zeros(1, vector.Size);
    for (var i = 0; i < vector.Size; i++) matrix[0, i] = vector.At(i);
    return matrix;
}

var oneFlower = split.TestX.Row(0);
var targetClass = (int)explainForest.Predict(AsRow(oneFlower)).At(0);

NdArray ClassProbability(NdArray batch)
{
    var probabilities = explainForest.PredictProbabilities(batch);
    var column = NdArray.Zeros(batch.Shape[0]);
    for (var i = 0; i < batch.Shape[0]; i++) column.SetAt(i, probabilities[i, targetClass]);
    return column;
}

var attribution = ShapleyValues.Sample(ClassProbability, oneFlower, split.TrainX, samples: 150);

Console.WriteLine($"explaining P({iris.LabelNames[targetClass]}) for one flower");
Console.WriteLine($"base value (average over the background) = {attribution.BaseValue:F4}\n");

foreach (var (feature, contribution) in attribution.Ranked)
    Console.WriteLine($"  {iris.FeatureNames[feature],-16} {contribution,+8:F4}");

Console.WriteLine($"\ncontributions sum to the prediction: {attribution.Prediction:F4}");
Console.WriteLine("efficiency holds exactly at any sample size, because each permutation telescopes");


## Calibration

A model can rank perfectly and still be badly calibrated. If everything it calls "90% likely"
happens 60% of the time, its ordering is fine and its numbers are not — and any decision made on a
threshold or an expected value is then wrong. Accuracy and AUC cannot see this.

Isotonic regression fixes it with a monotone remapping, so the ranking survives untouched.


In [ ]:
var calibrationRng = new GraviRandom(41);
var rawScores = NdArray.Zeros(4000);
var outcomes = NdArray.Zeros(4000);

for (var i = 0; i < 4000; i++)
{
    var truth = calibrationRng.NextDouble();
    rawScores.SetAt(i, Math.Sqrt(truth));    // well ranked, badly scaled
    outcomes.SetAt(i, calibrationRng.NextDouble() < truth ? 1 : 0);
}

var isotonic = new IsotonicRegression().Fit(rawScores, outcomes);
var fixedScores = isotonic.Predict(rawScores);

Console.WriteLine($"expected calibration error : {Calibration.ExpectedError(rawScores, outcomes):F4} -> {Calibration.ExpectedError(fixedScores, outcomes):F4}");
Console.WriteLine($"Brier score                : {Calibration.BrierScore(rawScores, outcomes):F4} -> {Calibration.BrierScore(fixedScores, outcomes):F4}");

var before = Calibration.Curve(rawScores, outcomes);
var after = Calibration.Curve(fixedScores, outcomes);

var reliability = new ScottPlot.Plot();
reliability.Add.Scatter(before.Select(p => p.MeanPredicted).ToArray(),
                        before.Select(p => p.ObservedFraction).ToArray()).LegendText = "before";
reliability.Add.Scatter(after.Select(p => p.MeanPredicted).ToArray(),
                        after.Select(p => p.ObservedFraction).ToArray()).LegendText = "after isotonic";

var ideal = reliability.Add.Scatter(new double[] { 0, 1 }, new double[] { 0, 1 });
ideal.LegendText = "perfectly calibrated";
ideal.LinePattern = ScottPlot.LinePattern.Dashed;

reliability.Title("Reliability diagram");
reliability.XLabel("mean predicted probability");
reliability.YLabel("observed frequency");
reliability.ShowLegend();
reliability.GetPngHtml(750, 500)


## HDBSCAN where no single eps works

Two tight clusters close together and one diffuse cluster far away. DBSCAN applies one `eps`
everywhere, so it has to choose which to get wrong — the sweep below shows it never gets all three.

HDBSCAN runs DBSCAN at *every* threshold at once and keeps the clusters that persist longest.


In [ ]:
using Gravicode.Science.GraviLearn.Clustering;

var densityRng = new GraviRandom(5);
var varied = NdArray.Zeros(150, 2);
for (var i = 0; i < 50; i++)   { varied[i, 0] = densityRng.Normal() * 0.3;     varied[i, 1] = densityRng.Normal() * 0.3; }
for (var i = 50; i < 100; i++) { varied[i, 0] = 3 + densityRng.Normal() * 0.3; varied[i, 1] = densityRng.Normal() * 0.3; }
for (var i = 100; i < 150; i++){ varied[i, 0] = 25 + densityRng.Normal() * 3.0; varied[i, 1] = 25 + densityRng.Normal() * 3.0; }

foreach (var epsilon in new[] { 0.5, 1.0, 2.0, 4.0 })
{
    var sweep = new Dbscan(epsilon, minSamples: 5);
    sweep.Fit(varied);

    var found = new HashSet<double>();
    var noise = 0;
    for (var i = 0; i < sweep.Labels.Size; i++)
    {
        if (sweep.Labels.At(i) < 0) noise++; else found.Add(sweep.Labels.At(i));
    }
    Console.WriteLine($"DBSCAN eps={epsilon,-5} -> {found.Count} clusters, {noise,3} noise");
}

var hdbscan = new Hdbscan(minClusterSize: 10).Fit(varied);
Console.WriteLine($"HDBSCAN            -> {hdbscan.ClusterCount} clusters, no threshold to choose");

var clusterPlot = new ScottPlot.Plot();
for (var label = -1; label < hdbscan.ClusterCount; label++)
{
    var xs = new List<double>();
    var ys = new List<double>();
    for (var i = 0; i < 150; i++)
        if ((int)hdbscan.Labels.At(i) == label) { xs.Add(varied[i, 0]); ys.Add(varied[i, 1]); }
    if (xs.Count == 0) continue;

    var series = clusterPlot.Add.Scatter(xs.ToArray(), ys.ToArray());
    series.LineWidth = 0;
    series.MarkerSize = 7;
    series.LegendText = label < 0 ? "noise" : $"cluster {label}";
}

clusterPlot.Title("HDBSCAN on clusters of differing density");
clusterPlot.ShowLegend();
clusterPlot.GetPngHtml(750, 500)


## Imbalanced data and novelty detection

On a dataset that is 92% negative, predicting "negative" everywhere scores 92%. SMOTE places new
minority points **between** real neighbours, so the classifier sees a region rather than a set of
dots — and class weights achieve the same rebalancing without touching the data at all.

A one-class SVM answers a different question: there are no negative examples to learn a boundary
*between*, so the task is to wrap the normal data as tightly as possible.


In [ ]:
using Gravicode.Science.GraviLearn.Resampling;
using Gravicode.Science.GraviLearn.Anomaly;

var imbalanceRng = new GraviRandom(7);
var skewedX = NdArray.Zeros(400, 2);
var skewedY = NdArray.Zeros(400);
for (var i = 0; i < 400; i++)
{
    var minority = i >= 370;
    skewedX[i, 0] = (minority ? 6 : 0) + imbalanceRng.Normal() * 0.6;
    skewedX[i, 1] = (minority ? 6 : 0) + imbalanceRng.Normal() * 0.6;
    skewedY.SetAt(i, minority ? 1 : 0);
}

foreach (var (label, count) in Resampler.ClassBalance(skewedY))
    Console.WriteLine($"class {label}: {count}");

Console.WriteLine($"\nOverSample -> {Resampler.OverSample(skewedX, skewedY).X.Shape[0]} rows");
Console.WriteLine($"SMOTE      -> {Resampler.Smote(skewedX, skewedY).X.Shape[0]} rows");

var classWeights = Resampler.ClassWeights(skewedY);
Console.WriteLine($"ClassWeights: {classWeights[0]:F4} / {classWeights[1]:F4}");
Console.WriteLine("Resample the TRAINING split only - doing it first leaks synthetic rows into the test set.\n");

var normalData = NdArray.Zeros(300, 2);
for (var i = 0; i < 300; i++)
{
    normalData[i, 0] = imbalanceRng.Normal();
    normalData[i, 1] = imbalanceRng.Normal();
}

var detector = new OneClassSvm(nu: 0.05).Fit(normalData);
var probes = NdArray.FromArray(new double[,] { { 0, 0 }, { 6, 6 }, { -8, 3 } });
var probeScores = detector.DecisionFunction(probes);

for (var i = 0; i < probes.Shape[0]; i++)
    Console.WriteLine($"({probes[i, 0],5:F1}, {probes[i, 1],5:F1}) score {probeScores.At(i),8:F4}  {(probeScores.At(i) >= 0 ? "normal" : "ANOMALY")}");


## Sparse training

A bag-of-words matrix is around 1% non-zero, and the dense copy is almost entirely zeros that cost
the same to store and multiply as any other number. `SparseLogisticRegression` trains on the CSR
form directly.

The claim worth making is *the same model, faster* — so the comparison below is between
**coefficients**, not accuracy. Accuracy would agree even if the weights had drifted.

In [ ]:
using Gravicode.Science.GraviLearn.Linear;

const int sparseDocs = 1500, sparseVocab = 3000;
var sparseRng = new GraviRandom(17);
var sparseTriplets = new List<(int Row, int Column, double Value)>();
var sparseLabels = NdArray.Zeros(sparseDocs);

for (var d = 0; d < sparseDocs; d++)
{
    var positive = d % 2 == 0;
    sparseLabels.SetAt(d, positive ? 1 : 0);
    for (var w = 0; w < 30; w++)
        sparseTriplets.Add((d, (int)(sparseRng.NextDouble() * sparseVocab), 1.0));
    sparseTriplets.Add((d, positive ? 0 : 1, 3.0));   // the two marker terms
}

var sparseX = SparseMatrix.FromTriplets(sparseDocs, sparseVocab, sparseTriplets);
Console.WriteLine($"{sparseX.NonZeroCount} non-zeros, {100.0 * sparseX.NonZeroCount / ((double)sparseDocs * sparseVocab):F2}% dense");

var sparseWatch = System.Diagnostics.Stopwatch.StartNew();
var sparseModel = new SparseLogisticRegression(learningRate: 1.0, maxIterations: 200).Fit(sparseX, sparseLabels);
sparseWatch.Stop();

var denseX = sparseX.ToDense();
var denseModel = new LogisticRegression(learningRate: 1.0, maxIterations: 200);
var denseWatch = System.Diagnostics.Stopwatch.StartNew();
denseModel.Fit(denseX, sparseLabels);
denseWatch.Stop();

var coefficientGap = 0.0;
for (var j = 0; j < sparseVocab; j++)
    coefficientGap = Math.Max(coefficientGap, Math.Abs(sparseModel.Coefficients[0, j] - denseModel.Coefficients.At(j)));

Console.WriteLine($"sparse {sparseWatch.ElapsedMilliseconds} ms, dense {denseWatch.ElapsedMilliseconds} ms");
Console.WriteLine($"largest coefficient difference: {coefficientGap:E2}");
Console.WriteLine($"markers: term 0 = {sparseModel.Coefficients[0, 0]:F4}, term 1 = {sparseModel.Coefficients[0, 1]:F4}");

In [ ]:
var denseMb = sparseDocs * (long)sparseVocab * 8 / (1024.0 * 1024.0);
var sparseMb = (sparseX.NonZeroCount * 12L + sparseDocs * 4L) / (1024.0 * 1024.0);
var shrink = denseMb / sparseMb;
var speedUp = denseWatch.Elapsed.TotalMilliseconds / sparseWatch.Elapsed.TotalMilliseconds;

// Ratios rather than raw MB and ms: those two share no unit and span three orders of magnitude,
// so on one linear axis the smaller vanishes and on a log axis a bar's length means nothing.
var sparsePlot = new ScottPlot.Plot();
sparsePlot.Add.Bars(new[] { 0.0, 1.0 }, new[] { shrink, speedUp });
sparsePlot.Add.Text($"{denseMb:F0} MB -> {sparseMb:F1} MB", 0.0, shrink * 1.04);
sparsePlot.Add.Text($"{denseWatch.ElapsedMilliseconds} ms -> {sparseWatch.ElapsedMilliseconds} ms", 1.0, speedUp * 1.04);
sparsePlot.Axes.Bottom.SetTicks(new[] { 0.0, 1.0 }, new[] { "smaller", "faster" });
sparsePlot.Axes.SetLimitsY(0, Math.Max(shrink, speedUp) * 1.25);
sparsePlot.YLabel("times better than the dense path");
sparsePlot.Title("Identical logistic model, sparse versus dense");
sparsePlot.GetPngHtml(800, 500)

## Distributed training

`DistributedForest` is **bit-identical** to single-process training, not merely equivalent: tree
`t` is seeded from `seed + t * 7919`, a function of its global index alone, so a shard boundary
cannot change the answer.

The weighting below is a correctness requirement rather than a refinement. A plain average of
per-worker gradients equals the global gradient only when every shard is the same size.

In [ ]:
using Gravicode.Science.GraviLearn.Distributed;

foreach (var shard in DataParallel.Partition(items: 1000, workers: 7))
    Console.WriteLine($"  {shard,-14} n={shard.Count}");

// Unequal shards with different local gradients - what happens when the data is not shuffled.
var shardValues = NdArray.Zeros(1000);
var shardRng = new GraviRandom(3);
for (var i = 0; i < 1000; i++) shardValues.SetAt(i, i / 100.0 + shardRng.Normal() * 0.5);

var unevenShards = new[] { new Shard(0, 900), new Shard(900, 100) };
var workerGradients = new List<NdArray>();
var workerCounts = new List<int>();
foreach (var shard in unevenShards)
{
    var total = 0.0;
    for (var i = shard.Start; i < shard.End; i++) total += shardValues.At(i);
    workerGradients.Add(NdArray.FromValues([total / shard.Count]));
    workerCounts.Add(shard.Count);
}

var trueGradient = 0.0;
for (var i = 0; i < 1000; i++) trueGradient += shardValues.At(i);
trueGradient /= 1000;

Console.WriteLine($"true gradient      : {trueGradient:F6}");
Console.WriteLine($"weighted average   : {DataParallel.AverageGradients(workerGradients, workerCounts).At(0):F6}");
Console.WriteLine($"unweighted average : {workerGradients.Sum(g => g.At(0)) / workerGradients.Count:F6}   <- wrong");

In [ ]:
var distributedForest = new DistributedForest(nTrees: 120, maxDepth: 10, seed: 42)
    .Fit(split.TrainX, split.TrainY, workers: 4);
var singleForest = new DistributedForest(nTrees: 120, maxDepth: 10, seed: 42)
    .Fit(split.TrainX, split.TrainY, workers: 1);

var manyWorkers = distributedForest.Predict(split.TestX);
var oneWorker = singleForest.Predict(split.TestX);

var bitIdentical = true;
for (var i = 0; i < manyWorkers.Size; i++) bitIdentical &= manyWorkers.At(i) == oneWorker.At(i);

Console.WriteLine($"4 workers accuracy : {distributedForest.Score(split.TestX, split.TestY):P2}");
Console.WriteLine($"1 worker  accuracy : {singleForest.Score(split.TestX, split.TestY):P2}");
Console.WriteLine($"bit-identical      : {bitIdentical}");
Console.WriteLine("Iris is 105 training rows, so coordination costs more than the trees do - what is");
Console.WriteLine("shown here is the identity, not a speed-up.");